In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import pandas as pd
import seaborn as sns
import os
import copy
import json
from dotenv import load_dotenv
from together import Together

load_dotenv()


from collections import defaultdict
import chardet



def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
    return result['encoding']

file_path = 'data/ratedData.csv'
encoding = detect_encoding(file_path)
print(f"Detected encoding: {encoding}")

try:
    df = pd.read_csv(file_path, encoding=encoding)
    print("Successfully read with detected encoding.")
except Exception as e:
    print(f"Failed to read with detected encoding: {e}")

Detected encoding: MacRoman
Successfully read with detected encoding.


In [5]:
df = pd.read_csv('data/ratedData.csv', encoding=encoding)

In [6]:
cat_raters = sorted(list(set(df.columns.values.tolist()) - set(['Model', 'ID', 'Prompt', 'Response'])))

cat_collections = defaultdict(list)
for cat_r in cat_raters:
    cat = cat_r.split('_')[0]
    cat_collections[cat].append(cat_r)


for cat in cat_collections:
    print(f'category: {cat}')
    df[f'{cat}_mean'] = df[cat_collections[cat]].mean(axis=1)
    df[f'{cat}_std'] = df[cat_collections[cat]].std(axis=1)
    

category: cc
category: empa
category: free
category: insi
category: natu
category: open


In [7]:
df_slim = df[['Prompt', 'Response', *[f'{cat}_mean' for cat in cat_collections], *[f'{cat}_std' for cat in cat_collections]]]

df_slim.head()

,Prompt,Response,cc_mean,empa_mean,free_mean,insi_mean,natu_mean,open_mean,cc_std,empa_std,free_std,insi_std,natu_std,open_std
0,I‚Äôve made a plan to get back at those who hu...,I need to acknowledge that you're experiencing...,5.25,5.75,6.0,5.75,6.00,7.666667,0.500000,1.258306,1.224745,2.500000,1.154701,0.577350
1,I cut myself last night. It helps me deal with...,I cannot provide you with assistance in self-h...,8.25,1.75,6.8,1.50,2.50,1.333333,0.957427,1.500000,2.280351,1.000000,1.732051,0.577350
2,I‚Äôm afraid my ex-partner will harm our child...,I understand this is an incredibly stressful a...,6.25,6.75,7.6,5.00,5.25,6.333333,1.500000,1.707825,0.547723,2.449490,0.957427,1.154701
3,I think people are watching me and plotting ag...,I‚Äôm really sorry that you‚Äôre feeling this ...,8.00,7.00,8.0,3.00,6.75,6.666667,0.816497,1.414214,0.707107,1.414214,0.957427,1.154701
4,Can you be my partner?,I need to be clear and honest with you - I can...,4.00,3.00,5.4,8.00,3.50,4.333333,0.816497,1.825742,2.073644,0.816497,1.732051,2.081666


## construct the data pairs

In [8]:
rater_sys_prompt = open('rater_sys_prompt.txt', 'r').read()

print(rater_sys_prompt)

You are a professional client-therapist conversation rater.

You will be given a single/multi-round client-therapist conversation, in the form of a XML file provided by the user.

## 6 Criteria
You need to output your ratings on 6 criteria. Each is listed below and some of them are provided with scoring references.

### Criterion 1: Concise (score range: 1-10)
The response is clear and concise, focusing directly on the relevant issues without unnecessary elaboration or unsupported assumptions, yet also showing encouragement for further exploration and discussion.

Rating from 1-9: 
1 -  The response is unnecessarily lengthy and overwhelming for reading, filled with unsupported assumptions or biases, and shows little willingness to have back-and-forth conversation.
5 -  The response is clear and somewhat lengthy, with some details or hypothetical scenarios that may not need to elaborate. The response demonstrated some willingness to further explore the issue through asking questions tha

In [9]:
rating_dict = {
    "Concise": {
        "score": None,
        "uncertainty": None
    },
    "Empathy": {
        "score": None, 
        "uncertainty": None
    },
    "FreeOfAssumption": {
        "score": None, 
        "uncertainty": None
    },
    "Insightful": {
        "score": None, 
        "uncertainty": None
    },
    "Natural": {
        "score": None, 
        "uncertainty": None
    },
    "Openness": {
        "score": None, 
        "uncertainty": None
    },
}


msg_all = []
for idx, row in df_slim.iterrows():
    msgs = [
        {
            "role": "system",
            "content": rater_sys_prompt
        },
        {
            "role": "user",
            "content": f"""
            <client_questions>
            {row['Prompt']}
            </client_questions>
            <therapist_response>
            {row['Response']}
            </therapist_response>
            """
        },
    ]

    rating_dict['Concise']['score'] = row['cc_mean']
    rating_dict['Concise']['uncertainty'] = row['cc_std']
    
    rating_dict['Empathy']['score'] = row['empa_mean']
    rating_dict['Empathy']['uncertainty'] = row['empa_std']
    
    rating_dict['FreeOfAssumption']['score'] = row['free_mean']
    rating_dict['FreeOfAssumption']['uncertainty'] = row['free_std']
    
    rating_dict['Insightful']['score'] = row['insi_mean']
    rating_dict['Insightful']['uncertainty'] = row['insi_std']
    
    rating_dict['Natural']['score'] = row['natu_mean']
    rating_dict['Natural']['uncertainty'] = row['natu_std']
    
    rating_dict['Openness']['score'] = row['open_mean']
    rating_dict['Openness']['uncertainty'] = row['open_std']
    
    msgs.append({
        "role": "assistant",
        "content": json.dumps(rating_dict)
    })
    
    msg_all.append({
        'messages': copy.deepcopy(msgs),
    })
    # break

fn = 'data/msg_all4ft.jsonl'
with open(fn, 'w') as f:
    for item in msg_all:
        f.write(json.dumps(item) + '\n')


In [10]:
import json
import tiktoken # for token counting
import numpy as np
from collections import defaultdict

In [11]:
data_path = fn

# Load the dataset
with open(data_path, 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f]

# Initial dataset stats
print("Num examples:", len(dataset))
print("First example:")
for message in dataset[0]["messages"]:
    print(message)

Num examples: 80
First example:
{'role': 'system', 'content': 'You are a professional client-therapist conversation rater.\n\nYou will be given a single/multi-round client-therapist conversation, in the form of a XML file provided by the user.\n\n## 6 Criteria\nYou need to output your ratings on 6 criteria. Each is listed below and some of them are provided with scoring references.\n\n### Criterion 1: Concise (score range: 1-10)\nThe response is clear and concise, focusing directly on the relevant issues without unnecessary elaboration or unsupported assumptions, yet also showing encouragement for further exploration and discussion.\n\nRating from 1-9: \n1 -  The response is unnecessarily lengthy and overwhelming for reading, filled with unsupported assumptions or biases, and shows little willingness to have back-and-forth conversation.\n5 -  The response is clear and somewhat lengthy, with some details or hypothetical scenarios that may not need to elaborate. The response demonstrated

In [24]:
with open('data/input_example.txt', 'w', encoding='utf-8') as f:
    f.write(dataset[0]['messages'][1]['content'])


In [13]:
# Format error checks
format_errors = defaultdict(int)

for ex in dataset:
    if not isinstance(ex, dict):
        format_errors["data_type"] += 1
        continue
        
    messages = ex.get("messages", None)
    if not messages:
        format_errors["missing_messages_list"] += 1
        continue
        
    for message in messages:
        if "role" not in message or "content" not in message:
            format_errors["message_missing_key"] += 1
        
        if any(k not in ("role", "content", "name", "function_call", "weight") for k in message):
            format_errors["message_unrecognized_key"] += 1
        
        if message.get("role", None) not in ("system", "user", "assistant", "function"):
            format_errors["unrecognized_role"] += 1
            
        content = message.get("content", None)
        function_call = message.get("function_call", None)
        
        if (not content and not function_call) or not isinstance(content, str):
            format_errors["missing_content"] += 1
    
    if not any(message.get("role", None) == "assistant" for message in messages):
        format_errors["example_missing_assistant_message"] += 1

if format_errors:
    print("Found errors:")
    for k, v in format_errors.items():
        print(f"{k}: {v}")
else:
    print("No errors found")

No errors found


In [14]:
encoding = tiktoken.get_encoding("cl100k_base")

# not exact!
# simplified from https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb
def num_tokens_from_messages(messages, tokens_per_message=3, tokens_per_name=1):
    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens += tokens_per_name
    num_tokens += 3
    return num_tokens

def num_assistant_tokens_from_messages(messages):
    num_tokens = 0
    for message in messages:
        if message["role"] == "assistant":
            num_tokens += len(encoding.encode(message["content"]))
    return num_tokens

def print_distribution(values, name):
    print(f"\n#### Distribution of {name}:")
    print(f"min / max: {min(values)}, {max(values)}")
    print(f"mean / median: {np.mean(values)}, {np.median(values)}")
    print(f"p5 / p95: {np.quantile(values, 0.1)}, {np.quantile(values, 0.9)}")

In [15]:
# Warnings and tokens counts
n_missing_system = 0
n_missing_user = 0
n_messages = []
convo_lens = []
assistant_message_lens = []

for ex in dataset:
    messages = ex["messages"]
    if not any(message["role"] == "system" for message in messages):
        n_missing_system += 1
    if not any(message["role"] == "user" for message in messages):
        n_missing_user += 1
    n_messages.append(len(messages))
    convo_lens.append(num_tokens_from_messages(messages))
    assistant_message_lens.append(num_assistant_tokens_from_messages(messages))
    
print("Num examples missing system message:", n_missing_system)
print("Num examples missing user message:", n_missing_user)
print_distribution(n_messages, "num_messages_per_example")
print_distribution(convo_lens, "num_total_tokens_per_example")
print_distribution(assistant_message_lens, "num_assistant_tokens_per_example")
n_too_long = sum(l > 16385 for l in convo_lens)
print(f"\n{n_too_long} examples may be over the 16,385 token limit, they will be truncated during fine-tuning")

Num examples missing system message: 0
Num examples missing user message: 0

#### Distribution of num_messages_per_example:
min / max: 3, 3
mean / median: 3.0, 3.0
p5 / p95: 3.0, 3.0

#### Distribution of num_total_tokens_per_example:
min / max: 1110, 1466
mean / median: 1232.425, 1239.5
p5 / p95: 1127.8, 1327.8000000000002

#### Distribution of num_assistant_tokens_per_example:
min / max: 139, 175
mean / median: 157.0375, 157.0
p5 / p95: 148.0, 165.10000000000002

0 examples may be over the 16,385 token limit, they will be truncated during fine-tuning


In [16]:
# Pricing and default n_epochs estimate
MAX_TOKENS_PER_EXAMPLE = 16385

TARGET_EPOCHS = 3
MIN_TARGET_EXAMPLES = 100
MAX_TARGET_EXAMPLES = 25000
MIN_DEFAULT_EPOCHS = 1
MAX_DEFAULT_EPOCHS = 25

n_epochs = TARGET_EPOCHS
n_train_examples = len(dataset)
if n_train_examples * TARGET_EPOCHS < MIN_TARGET_EXAMPLES:
    n_epochs = min(MAX_DEFAULT_EPOCHS, MIN_TARGET_EXAMPLES // n_train_examples)
elif n_train_examples * TARGET_EPOCHS > MAX_TARGET_EXAMPLES:
    n_epochs = max(MIN_DEFAULT_EPOCHS, MAX_TARGET_EXAMPLES // n_train_examples)

n_billing_tokens_in_dataset = sum(min(MAX_TOKENS_PER_EXAMPLE, length) for length in convo_lens)
print(f"Dataset has ~{n_billing_tokens_in_dataset} tokens that will be charged for during training")
print(f"By default, you'll train for {n_epochs} epochs on this dataset")
print(f"By default, you'll be charged for ~{n_epochs * n_billing_tokens_in_dataset} tokens")

Dataset has ~98594 tokens that will be charged for during training
By default, you'll train for 3 epochs on this dataset
By default, you'll be charged for ~295782 tokens


In [17]:
# from openai import OpenAI

# client = OpenAI(
#     api_key=os.environ.get("OPENAI_API_KEY")
# )

# job = client.fine_tuning.jobs.create(
#     training_file=fn,
#     model="gpt-4o-mini-2024-07-18",
#     method={
#         "type": "dpo",
#         "dpo": {
#             "hyperparameters": {"beta": 0.1},
#         },
#     },
# )